# 과제 4 — 데이터 증강 성능 비교

## 가상 이미지 데이터셋 생성

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

# 가상 이미지 데이터셋 — PyTorch 형식 (N, C, H, W)
X = np.random.random((1000, 3, 32, 32)).astype(np.float32)
y = np.random.randint(2, size=1000)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'학습: {X_train_np.shape}, 테스트: {X_test_np.shape}')

## 공통 모델 정의

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.transform:
            x = self.transform(x)
        return x, self.y[idx]


def build_model():
    return nn.Sequential(
        nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(32 * 16 * 16, 64), nn.ReLU(),
        nn.Linear(64, 2)
    )


def train_and_eval(model, train_loader, test_loader, epochs=10):
    optimizer = optim.Adam(model.parameters())
    criterion = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for X_b, y_b in train_loader:
            optimizer.zero_grad()
            criterion(model(X_b), y_b).backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        correct = sum((model(X_b).argmax(1) == y_b).sum().item()
                      for X_b, y_b in test_loader)
    return correct / len(test_loader.dataset)

## 증강 없이 학습

In [ ]:
test_ds    = ImageDataset(X_test_np, y_test_np)
test_ldr   = DataLoader(test_ds, batch_size=32)

train_ds_base  = ImageDataset(X_train_np, y_train_np)
train_ldr_base = DataLoader(train_ds_base, batch_size=32, shuffle=True)

model_base = build_model()
acc_base   = train_and_eval(model_base, train_ldr_base, test_ldr)
print(f'\n증강 없이 — 테스트 정확도: {acc_base:.4f}')

## 증강 적용 후 학습

In [ ]:
aug_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomAffine(degrees=20, translate=(0.1, 0.1)),
])

train_ds_aug  = ImageDataset(X_train_np, y_train_np, transform=aug_transform)
train_ldr_aug = DataLoader(train_ds_aug, batch_size=32, shuffle=True)

model_aug = build_model()
acc_aug   = train_and_eval(model_aug, train_ldr_aug, test_ldr)
print(f'\n증강 적용 — 테스트 정확도: {acc_aug:.4f}')

## 성능 비교

In [ ]:
print(f'증강 없이:  {acc_base:.4f}')
print(f'증강 적용:  {acc_aug:.4f}')
print(f'차이:       {acc_aug - acc_base:+.4f}')